# Mars · 04 — XGBoost inference (5-feature tabular)

**Primary interface** for tabular inference. This notebook calls the shared
`channel_heads.inference` package (`load_feature_columns`, `load_threshold`,
`load_xgb_model`, `verify_feature_matrix`, `predict_with_threshold`) — the same
functions the batch wrapper `scripts/run_mars_xgb_inference_5feat.py` uses.

It is **read-only**: it scores the model-ready feature table in memory and
summarises the result. It writes no prediction files — those are produced by the
batch wrapper.

> The trained model artifacts live under `models/` (git-ignored). If they are
> not present, the prediction cell below reports that and stops gracefully; the
> notebook is fully functional once the artifacts exist.

In [1]:
import pandas as pd

from channel_heads.io.paths import PROJECT_ROOT
from channel_heads.models.xgboost import (
    load_feature_columns,
    load_threshold,
    load_xgb_model,
    predict_with_threshold,
    verify_feature_matrix,
)

READY_PARQUET = PROJECT_ROOT / "data/Mars/model_inputs/mars_pair_features_5feat_model_ready.parquet"
MODEL_PATH = PROJECT_ROOT / "models/xgb_touching_classifier.json"
FEATURE_COLUMNS_TXT = PROJECT_ROOT / "models/feature_columns.txt"
OPTIMAL_THRESHOLD_TXT = PROJECT_ROOT / "models/optimal_threshold.txt"

artifacts = [MODEL_PATH, FEATURE_COLUMNS_TXT, OPTIMAL_THRESHOLD_TXT]
have_model = all(p.exists() for p in artifacts)
print("features table exists:", READY_PARQUET.exists())
print("model artifacts present:", have_model)

features table exists: True
model artifacts present: False


## Load the model-ready feature table (read-only)

In [2]:
df = pd.read_parquet(READY_PARQUET)
print(f"{len(df)} model-ready pairs x {df.shape[1]} columns")

3785 model-ready pairs x 28 columns


## Score with the production model — via the package

Loads the model + authoritative feature order + decision threshold, runs the
shared pre-prediction checks, then predicts. Skips gracefully if the artifacts
are absent.

In [3]:
if not have_model:
    missing = [str(p.relative_to(PROJECT_ROOT)) for p in artifacts if not p.exists()]
    print("Model artifacts absent — skipping prediction. Missing:")
    for m in missing:
        print("  -", m)
    print("Train/restore the model, or run scripts/run_mars_xgb_inference_5feat.py.")
else:
    model_features = load_feature_columns(FEATURE_COLUMNS_TXT)
    threshold = load_threshold(OPTIMAL_THRESHOLD_TXT)
    stats = verify_feature_matrix(df, model_features)
    print(f"feature matrix {stats['n_rows']}x{stats['n_cols']}, "
          f"NaN cells={stats['n_nan_cells']} (handled natively)")

    model = load_xgb_model(MODEL_PATH, expected_features=model_features)
    proba, pred = predict_with_threshold(model, df, model_features, threshold)
    print(f"threshold={threshold:.6f}")
    print(f"predicted touching: {int(pred.sum())}/{len(pred)} ({100*pred.mean():.1f}%)")
    print("prob summary:")
    print(pd.Series(proba).describe()[["min", "25%", "50%", "mean", "75%", "max"]].round(4))

Model artifacts absent — skipping prediction. Missing:
  - models/xgb_touching_classifier.json
  - models/feature_columns.txt
  - models/optimal_threshold.txt
Train/restore the model, or run scripts/run_mars_xgb_inference_5feat.py.


---
Full-dataset run (writes predictions, GeoPackage layers and summary figures):

```bash
python scripts/run_mars_xgb_inference_5feat.py
```